# MOTION-AI: End-to-End Machine Learning Pipeline

This notebook covers **Milestones 4 through 7**:
1. Data Cleaning & Preprocessing (Milestone 4)
2. Exploratory Data Analysis (Milestone 5)
3. Baseline Model Training & Evaluation (Milestone 6)
4. Candidate Models Benchmarking (Milestone 7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

## 1. Data Loading (Milestone 3)

In [ ]:
df = pd.read_csv('../../data/predictive_maintenance_dataset.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())

## 2. Exploratory Data Analysis (Milestone 5)

In [ ]:
# Class Imbalance Check
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Target', palette='viridis')
plt.title('Distribution of Failures (Target)')
plt.show()
print(df['Target'].value_counts(normalize=True))

In [ ]:
# Feature Correlation Heatmap
numeric_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Target']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

## 3. Data Cleaning & Preprocessing (Milestone 4)

In [ ]:
# Drop identifiers
X = df.drop(columns=['UDI', 'Product ID', 'Target', 'Type'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

## 4. Baseline Model: Logistic Regression (Milestone 6)

In [ ]:
baseline = LogisticRegression(class_weight='balanced')
baseline.fit(X_train_scaled, y_train)
y_pred_base = baseline.predict(X_test_scaled)

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}\n")
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

res_base = evaluate_model(y_test, y_pred_base, "Baseline (Logistic Regression)")

## 5. Candidate Models Benchmarking (Milestone 7)

In [ ]:
# Model 1: Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
res_rf = evaluate_model(y_test, y_pred_rf, "Random Forest")

# Model 2: XGBoost
xgb = XGBClassifier(scale_pos_weight=(len(y_train) - sum(y_train))/sum(y_train), random_state=42)
xgb.fit(X_train_scaled, y_train)
y_pred_xgb = xgb.predict(X_test_scaled)
res_xgb = evaluate_model(y_test, y_pred_xgb, "XGBoost")

In [ ]:
results_df = pd.DataFrame([res_base, res_rf, res_xgb]).set_index('Model')
display(results_df.style.background_gradient(cmap='Greens'))

results_df[['F1', 'Recall']].plot(kind='bar', figsize=(8, 5))
plt.title('Model Comparison: F1 Score & Recall')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()